In [1]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor


DATA_FOLDER = "./"

df = pd.read_pickle(DATA_FOLDER + "df_fe_epic_light_best_customers_50c.pickle")
product_ids = pd.read_csv(DATA_FOLDER + "product_id_apredecir201912.txt", sep="\t")[
    "product_id"
].tolist()
#df = df[df["product_id"].isin(product_ids)]
df.drop(columns=["periodo"], inplace=True, errors="ignore")

/home/fede/.venvs/labo3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
numeric_df = df.select_dtypes(include=[np.number])
total_infs = np.isinf(numeric_df.values).sum()
print(f"Total infs: {total_infs}")
df[numeric_df.columns] = df[numeric_df.columns].replace([np.inf, -np.inf], np.nan)

Total infs: 0


In [3]:
df['fecha'] = df['fecha'].apply(lambda x: x.to_timestamp('M'))  # último día del mes
df

,product_id,customer_id,fecha,periodo_min_producto,periodo_max_producto,periodo_min_customer,periodo_max_customer,plan_precios_cuidados,cust_request_qty,cust_request_tn,...,cust_request_qty_sku_size_vendidas_div,cust_request_qty_product_id_vendidas,cust_request_qty_product_id_vendidas_div,cust_request_qty_customer_id_vendidas,cust_request_qty_customer_id_vendidas_div,tn_customer_vendidas,tn_total_vendidas,tn_customer_weight,tn_product_vendidas,tn_product_weight
0,20001,0,2017-01-31,NaT,NaT,NaT,NaT,NaN,196,94.871902,...,0.036745,479,0.409186,80580,0.002432,7129.603516,34057.316406,0.209341,934.772217,2.744703e-02
1,20001,10001,2017-01-31,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,11,99.438606,...,0.002062,479,0.022965,4013,0.002741,2543.899414,34057.316406,0.074695,934.772217,2.744703e-02
2,20001,10002,2017-01-31,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,17,38.683010,...,0.003187,479,0.035491,14680,0.001158,3143.644775,34057.316406,0.092305,934.772217,2.744703e-02
3,20001,10003,2017-01-31,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,17,143.494263,...,0.003187,479,0.035491,8182,0.002078,2425.989014,34057.316406,0.071233,934.772217,2.744703e-02
4,20001,10004,2017-01-31,2017-01-01,2019-12-01,2017-01-01,2019-12-01,0.0,9,184.729263,...,0.001687,479,0.018789,5606,0.001605,1738.918335,34057.316406,0.051059,934.772217,2.744703e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1607617,21276,10046,2019-12-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,0.000000,4,0.000000,65,0.000000,56.943310,26217.066406,0.002172,0.008920,3.402363e-07
1607618,21276,10047,2019-12-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,0.000000,4,0.000000,135,0.000000,94.281326,26217.066406,0.003596,0.008920,3.402363e-07
1607619,21276,10048,2019-12-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,0.000000,4,0.000000,246,0.000000,166.782974,26217.066406,0.006362,0.008920,3.402363e-07
1607620,21276,10050,2019-12-31,2019-03-01,2019-12-01,2017-01-01,2019-12-01,0.0,0,0.000000,...,0.000000,4,0.000000,80,0.000000,117.360664,26217.066406,0.004476,0.008920,3.402363e-07


In [4]:
df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)

/tmp/ipykernel_182621/2957026289.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["serie_id"] = df["product_id"].astype(str) + "_" + df["customer_id"].astype(str)


In [5]:
TEST_DATE = 33

def get_indexes(df, test_date=TEST_DATE):
    test_index = df.index[df['date_id'] == test_date]
    train_index = df.index[df['date_id'] <= test_date]
    train_scaler_index = df.index[df['date_id'] <= test_date]
    return test_index, train_index, train_scaler_index
df["target"] = df.groupby(['customer_id', 'product_id'])['tn'].shift(-2)

test_index, train_index, train_scaler_index = get_indexes(df)
train_df = df.loc[train_index].copy()
test_df = df.loc[test_index].copy()


/tmp/ipykernel_182621/3574717736.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df.groupby(['customer_id', 'product_id'])['tn'].shift(-2)


In [ ]:
# ahora agrego features:
# cat1, cat2, cat3, brand and sku_size show be in a static_features_df which contain product_id, and columns
static_features_df = pd.DataFrame({
    'cat1': train_df.groupby('serie_id')['cat1'].first(),
    'cat2': train_df.groupby('serie_id')['cat2'].first(),
    'cat3': train_df.groupby('serie_id')['cat3'].first(),
    'brand': train_df.groupby('serie_id')['brand'].first(),
    'sku_size': train_df.groupby('serie_id')['sku_size'].first(),
    "product_id": train_df.groupby('serie_id')['product_id'].first(),
    "customer_id": train_df.groupby('serie_id')['customer_id'].first(),
    "plan_precios_cuidados": train_df.groupby('serie_id')['plan_precios_cuidados'].first(),
}).reset_index()
# drop this from test)DF
train_df_customers = train_df.drop(columns=['cat1', 'cat2', 'cat3', 'brand', 'sku_size', "customer_id", "product_id", 'plan_precios_cuidados'])
static_features_df.isna().sum()
# cambviar dtypes int16 o int8 a int32 en train_df_no_static
for column in train_df_customers.columns:
    if train_df_customers[column].dtype == "int16" or train_df_customers[column].dtype == "int8":
        train_df_customers[column] = train_df_customers[column].astype("int32")    
# agrego la categoria "missing" y hago fill nan con missing
for column in static_features_df.columns:
    if static_features_df[column].dtype.name == "category":
        static_features_df[column] = static_features_df[column].cat.add_categories("missing")
    static_features_df[column] = static_features_df[column].fillna("missing").astype("category")
static_features_df.isna().sum()



train_data = TimeSeriesDataFrame.from_data_frame(
    train_df_customers.drop(columns=['target'], errors="ignore"),  # No necesito la columna target en train_data
    id_column="serie_id",
    timestamp_column="fecha",
    static_features_df=static_features_df
)
train_data.head()




predictor = TimeSeriesPredictor(
    prediction_length=2,
    target="tn",
    freq="ME",
    eval_metric="WAPE",  # Weighted Absolute Percentage Error
    #horizon_weight=[0.5, 1]
)


predictor.fit(
    train_data,
    #presets="fast_training",
    #presets=",
    num_val_windows=2,
    #time_limit=600,
)



Beginning AutoGluon training...
AutoGluon will save models to '/home/fede/programacion/labo3/AutogluonModels/ag-20250702_031736'
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #65-Ubuntu SMP PREEMPT_DYNAMIC Mon May 19 17:15:03 UTC 2025
CPU Count:          8
GPU Count:          0
Memory Avail:       6.26 GB / 15.32 GB (40.9%)
Disk Space Avail:   62.99 GB / 575.67 GB (10.9%)
Setting presets to: high_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': WAPE,
 'freq': 'ME',
 'hyperparameters': 'default',
 'known_covariates_names': [],
 'num_val_windows': 2,
 'prediction_length': 2,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'tn',
 'verbosity': 2}

train_data with frequency 'IRREG' has been resampled to 

In [7]:
test_pred = predictor.predict(train_data)
test_pred

data with frequency 'IRREG' has been resampled to frequency 'ME'.
Model not specified in predict, will default to the model with the best validation score: WeightedEnsemble


mean        0.1        0.2        0.3  \
item_id     timestamp                                                 
20001_0     2019-11-30  118.562772  69.749851  86.025546  97.405272   
            2019-12-31  117.689649  65.073429  83.036569  95.882337   
20001_10001 2019-11-30  129.464725  47.049356  67.566534  83.887844   
            2019-12-31  128.036752  43.908228  64.658501  81.321422   
20001_10002 2019-11-30   25.944032   2.103029   8.257959  13.139824   
...                            ...        ...        ...        ...   
21247_10048 2019-12-31    0.000013  -0.001805  -0.000282   0.000001   
21247_10050 2019-11-30    0.000006  -0.002975  -0.000935  -0.000032   
            2019-12-31    0.000013  -0.001805  -0.000282   0.000001   
21247_10051 2019-11-30    0.000006  -0.002975  -0.000935  -0.000032   
            2019-12-31    0.000013  -0.001805  -0.000282   0.000001   

                                 0.4         0.5         0.6         0.7  \
item_id     timestamp                                                      
20001_0     2019-11-30  1.081734e+02  118.562772  129.259401  141.307797   
            2019-12-31  1.070449e+02  117.689649  129.466613  141.514655   
20001_10001 2019-11-30  1.043323e+02  129.464725  157.983139  190.368933   
            2019-12-31  1.018986e+02  128.036752  158.033710  190.604382   
20001_10002 2019-11-30  1.895568e+01   25.944032   33.742412   42.879434   
...                              ...         ...         ...         ...   
21247_10048 2019-12-31  4.306350e-06    0.000013    0.000025    0.001093   
21247_10050 2019-11-30  8.133415e-07    0.000006    0.000012    0.000309   
            2019-12-31  4.306350e-06    0.000013    0.000025    0.001093   
21247_10051 2019-11-30  8.133415e-07    0.000006    0.000012    0.000309   
            2019-12-31  4.306350e-06    0.000013    0.000025    0.001093   

                               0.8         0.9  
item_id     timestamp                           
20001_0     2019-11-30  155.564187  177.900725  
            2019-12-31  156.428513  181.428813  
20001_10001 2019-11-30  232.096361  300.206166  
            2019-12-31  233.240758  304.715517  
20001_10002 2019-11-30   54.559036   73.687767  
...                            ...         ...  
21247_10048 2019-12-31    0.002839    0.004111  
21247_10050 2019-11-30    0.002037    0.003387  
            2019-12-31    0.002839    0.004111  
21247_10051 2019-11-30    0.002037    0.003387  
            2019-12-31    0.002839    0.004111  

[125358 rows x 10 columns]

In [15]:
predictions = test_pred.groupby('item_id').last()['mean'].reset_index()
predictions = predictions.rename(columns={'item_id': 'serie_id'})
predictions = predictions.merge(test_df[['serie_id',"product_id", 'target']].drop_duplicates(), on='serie_id', how='left')
predictions = predictions.groupby('product_id').agg({
    'mean': 'sum',
    'target': 'sum',
}).reset_index()
predictions["mean"] = predictions["mean"]
predictions = predictions[predictions["product_id"].isin(product_ids)]
predictions["abs_error"] = abs(predictions['target'] - predictions['mean'])
import numpy as np
total_error = np.abs(predictions['target'] - predictions['mean']).sum() / (predictions['target'].sum())
print(f"Total Absolute Error: {total_error:.4f}")
predictions

Total Absolute Error: 0.2883


,product_id,mean,target,abs_error
0,20001.0,1052.326303,1504.688599,452.362295
1,20002.0,764.722128,1087.308594,322.586466
2,20003.0,611.100422,892.501282,281.400860
3,20004.0,459.114723,637.900024,178.785301
4,20005.0,466.027533,593.244446,127.216913
...,...,...,...,...
947,21263.0,0.018749,0.012700,0.006049
949,21265.0,0.045548,0.050070,0.004522
950,21266.0,0.051640,0.051210,0.000430
951,21267.0,0.055662,0.015690,0.039972


best 0.2464 con fast training